In [0]:
from pyspark.sql import functions as F

plata = spark.table("workspace.plata.aplicaciones")
print("Filas leídas de plata:", plata.count())


Filas leídas de plata: 460


In [0]:
es_critica = F.when(F.col("aplicacion_critica") == "Crítica", 1).otherwise(0)

resumen = (
    plata.groupBy("administracion")
    .agg(
        F.count("*").alias("total"),
        F.sum(es_critica).alias("criticas"),
    )
    .withColumn("porcentaje_criticas", F.round(F.col("criticas") / F.col("total") * 100, 1))
    .orderBy(F.desc("total"))
)

resumen.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.oro.resumen_por_administracion")
resumen.show()


+--------------+-----+--------+-------------------+
|administracion|total|criticas|porcentaje_criticas|
+--------------+-----+--------+-------------------+
|          ACSI|   96|      64|               66.7|
|          APE1|   72|      31|               43.1|
|          APE5|   66|      10|               15.2|
|          AMCI|   62|       0|                0.0|
|          APE3|   62|      36|               58.1|
|          APE2|   41|      17|               41.5|
|         ACSMC|   39|       1|                2.6|
|   ACSMC - AIA|   19|       2|               10.5|
|         APCTI|    2|       0|                0.0|
|         ACDMA|    1|       0|                0.0|
+--------------+-----+--------+-------------------+



In [0]:
riesgo = (
    plata
    .filter((F.col("aplicacion_critica") == "Crítica") & (F.col("tipo_de_arquitectura") == "Legada"))
    .select(
        "clave_o_id",
        "acronimo_de_la_aplicacion",
        "nombre_completo_de_la_aplicacion",
        "administracion_general",
        "criticidad_remedy",
        "lenguaje_de_desarrollo",
        "manejador_y_version_de_la_base_de_datos",
    )
)

riesgo.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.oro.aplicaciones_riesgo")
print("Aplicaciones críticas y legadas:", riesgo.count())


Aplicaciones críticas y legadas: 69


In [0]:
display(spark.table("workspace.oro.riesgo_por_area") if False else
        spark.table("workspace.oro.aplicaciones_riesgo").groupBy("administracion_general").count().orderBy(F.desc("count")))


administracion_general,count
AGR,35
AGAFF,9
AGSC,9
AGCTI,7
AGGC,3
Por definir,3
AGP,2
AGE,1


In [0]:
busquedas = [
    ("Base de datos", "manejador_y_version_de_la_base_de_datos", ["Oracle", "Informix", "SQL Server", "DB2", "PostgreSQL", "MongoDB", "Access"]),
    ("Lenguaje", "lenguaje_de_desarrollo", ["Java", ".NET", "DataStage", "4GL", "Visual Basic", "ASP", "Cognos", "React"]),
]

filas = []
for tipo, columna, tecnologias in busquedas:
    for tecnologia in tecnologias:
        cantidad = plata.filter(F.lower(F.col(columna)).contains(tecnologia.lower())).count()
        filas.append((tipo, tecnologia, cantidad))

tecnologias_df = spark.createDataFrame(filas, ["tipo", "tecnologia", "aplicaciones"])
tecnologias_df = tecnologias_df.orderBy("tipo", F.desc("aplicaciones"))

tecnologias_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.oro.tecnologias")
display(tecnologias_df)


tipo,tecnologia,aplicaciones
Base de datos,Oracle,142
Base de datos,Informix,102
Base de datos,SQL Server,89
Base de datos,DB2,81
Base de datos,PostgreSQL,29
Base de datos,MongoDB,19
Base de datos,Access,4
Lenguaje,Java,183
Lenguaje,.NET,80
Lenguaje,DataStage,79


In [0]:
bronce = spark.table("workspace.bronce.matriz")

# Columnas a comparar: las de plata, menos fecha_carga
columnas = []
for columna in plata.columns:
    if columna != "fecha_carga":
        columnas.append(columna)


# Mide, para cada columna, cuántos valores distintos y cuántos nulos tiene
def medir(tabla, columnas):
    expresiones = []
    for columna in columnas:
        expresiones.append(F.countDistinct(columna).alias(columna + "__distintos"))
        expresiones.append(F.sum(F.when(F.col(columna).isNull(), 1).otherwise(0)).alias(columna + "__nulos"))
    return tabla.agg(*expresiones).collect()[0]


In [0]:
antes = medir(bronce, columnas)
despues = medir(plata, columnas)

filas = []
for columna in columnas:
    filas.append((
        columna,
        antes[columna + "__distintos"],
        despues[columna + "__distintos"],
        antes[columna + "__nulos"],
        despues[columna + "__nulos"],
    ))

calidad = spark.createDataFrame(
    filas,
    ["columna", "distintos_antes", "distintos_despues", "nulos_antes", "nulos_despues"],
)
calidad = calidad.withColumn("valores_unificados", F.col("distintos_antes") - F.col("distintos_despues"))
calidad = calidad.orderBy(F.desc("valores_unificados"))

calidad.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.oro.calidad_por_columna")
display(calidad)


columna,distintos_antes,distintos_despues,nulos_antes,nulos_despues,valores_unificados
severidad_remedy,10,4,0,10,6
sistema_operativo_middleware,110,104,1,1,6
manejador_y_version_de_la_base_de_datos,100,96,0,0,4
sistema_operativo_front_end,90,86,1,1,4
tipo_de_arquitectura,21,18,0,0,3
sistema_operativo_para_el_manejador_de_la_base_de_datos,77,74,1,1,3
pistas_auditoria,6,3,1,1,3
descripcion_pistas_de_auditoria,123,120,1,1,3
mesa_de_servicio,6,3,1,1,3
administracion_central,17,15,0,0,2
